# AOD EDA: VIIRS 0.55 µm (NOAA-20) vs Himawari (0.50 µm original & 0.55 µm AE-corrected)

Exploratory Data Analysis comparing Aerosol Optical Depth (AOD) measurements from:
- **VIIRS NOAA-20** — `aod` column at 0.55 µm (nearest-pixel, raw L2 matchup)
- **Himawari L3 Envisoft** — two variants:
  - `AOD_Himawari` : original `AOT_Merged_center` at 0.50 µm
  - `AOD_Himawari_055` : wavelength-corrected to 0.55 µm using the Ångström exponent (`AE_Merged`) when available; falls back to the original 0.50 µm value when `AE_Merged` is NaN

**Wavelength conversion formula** (when AE is available):
$$\tau_{0.55} = \tau_{0.50} \times \left(\frac{0.55}{0.50}\right)^{-\alpha}$$

**VIIRS QA filtering** (applied before analysis):
- `qa == 0` → no valid retrieval (already NaN in `aod`)
- `qa >= 1` → valid; kept by default
- `qa == 3` → highest confidence; use `VIIRS_QA_MIN = 3` to be strict

Goals:
1. Load VIIRS flat CSV and Himawari per-station CSVs; match stations via masterdata coordinates
2. Merge on overlapping timestamps (±1 h window); compute `AOD_Himawari_055`
3. Correlation analysis — VIIRS 0.55 µm vs Himawari 0.50 µm (original) **and** vs Himawari 0.55 µm (AE-corrected)
4. Time-series comparison per station
5. Aggregate statistics across all stations

## 0 · Imports & Configuration

In [ ]:
import os
import warnings
import string
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')

# ── Academic style ──────────────────────────────────────────────────────────
sns.set_theme(style='ticks', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 300,
    'figure.facecolor': 'white',
    'font.family': 'serif',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'axes.titlesize': 14
})

# ── Paths ───────────────────────────────────────────────────────────────────
VIIRS_CSV  = Path('/home/slow_data/Air_Quality/VIIRS/output/viirs_aod_L2_NOAA20_nearest_raw.csv')
HIMA_DIR   = Path('/home/slow_data/Air_Quality/Himawari/station_aod_v3/L3')
MASTERDATA = Path('/home/work1/projects/Air_Quality/Masterdata/envisoft_27_stations.csv')

# ── QA threshold for VIIRS ──────────────────────────────────────────────────
# 1 = keep all valid retrievals; 3 = keep only highest-confidence pixels
VIIRS_QA_MIN = 1

# ── Column names used throughout ────────────────────────────────────────────
VIIRS_COL    = 'AOD_VIIRS_055'       # VIIRS AOD at 0.55 µm
HIMA_050_COL = 'AOD_Himawari'        # Himawari original 0.50 µm
HIMA_055_COL = 'AOD_Himawari_055'    # Himawari AE-corrected to 0.55 µm

# Convenience list: (col, short label, colour)
HIMA_VARIANTS = [
    (HIMA_050_COL, 'Himawari 0.50 µm (original)',      '#2D7DBB'),
    (HIMA_055_COL, 'Himawari 0.55 µm (AE-corrected)', '#1B4F8A'),
]

WL_RATIO = 0.55 / 0.50   # wavelength ratio for Ångström conversion

# ── Load masterdata ─────────────────────────────────────────────────────────
_meta = pd.read_csv(MASTERDATA)

print(f'Masterdata loaded : {len(_meta)} stations')
print(f'VIIRS file exists : {VIIRS_CSV.exists()}')
print(f'Himawari dir exists: {HIMA_DIR.exists()}')

## 1 · Load VIIRS Data & Discover Himawari Station Files

In [ ]:
# ── Load full VIIRS CSV ─────────────────────────────────────────────────────
viirs_raw = pd.read_csv(VIIRS_CSV, parse_dates=['datetime'], low_memory=False)
viirs_raw['datetime'] = pd.to_datetime(viirs_raw['datetime'])
viirs_raw = viirs_raw.dropna(subset=['datetime'])

# Apply QA filter — keep only rows where qa >= VIIRS_QA_MIN AND aod is not NaN
viirs_raw = viirs_raw[viirs_raw['qa'] >= VIIRS_QA_MIN].copy()
viirs_raw = viirs_raw.dropna(subset=['aod'])

print(f'VIIRS rows after QA≥{VIIRS_QA_MIN} + non-NaN aod : {len(viirs_raw):,}')
print(f'Date range : {viirs_raw["datetime"].min().date()} → {viirs_raw["datetime"].max().date()}')
print(f'Unique stations in VIIRS CSV : {viirs_raw["station"].nunique()}')
print('\nQA value counts:')
print(viirs_raw['qa'].value_counts().sort_index())
print('\nSource SDS breakdown:')
print(viirs_raw['source_sds'].value_counts())

In [ ]:
# ── Match VIIRS station names → masterdata stationName via coordinates ───────
# VIIRS CSV provides station_lat / station_lon; masterdata has Latitude / Longitude.
# We snap each unique VIIRS (station_lat, station_lon) pair to the nearest masterdata row.

COORD_TOL_DEG = 0.01   # ~1 km tolerance for coordinate matching

# Build a lookup: unique (station_lat, station_lon) from VIIRS
viirs_stations = (
    viirs_raw[['station', 'station_lat', 'station_lon']]
    .drop_duplicates(subset='station')
    .reset_index(drop=True)
)

# Detect lat/lon column names in masterdata (flexible)
lat_col = next((c for c in _meta.columns if 'lat' in c.lower()), None)
lon_col = next((c for c in _meta.columns if 'lon' in c.lower()), None)
name_col = 'stationName'   # assumed — adjust if your masterdata uses a different name

print(f'Masterdata columns : {list(_meta.columns)}')
print(f'Using lat={lat_col}, lon={lon_col}, name={name_col}')

station_id_map = {}   # viirs station label → masterdata stationName (= Himawari file stem)

for _, row in viirs_stations.iterrows():
    v_lat, v_lon = row['station_lat'], row['station_lon']
    dist = np.sqrt((_meta[lat_col] - v_lat)**2 + (_meta[lon_col] - v_lon)**2)
    idx_min = dist.idxmin()
    if dist[idx_min] <= COORD_TOL_DEG:
        station_id_map[row['station']] = _meta.loc[idx_min, name_col]

print(f'\nVIIRS stations matched to masterdata : {len(station_id_map)} / {len(viirs_stations)}')
if len(station_id_map) < len(viirs_stations):
    unmatched = viirs_stations[~viirs_stations['station'].isin(station_id_map)]['station'].tolist()
    print(f'Unmatched stations ({len(unmatched)}): {unmatched[:5]} ...')
    print(f'  → Increase COORD_TOL_DEG (currently {COORD_TOL_DEG}) if too many stations are unmatched.')

In [ ]:
# ── Discover Himawari per-station CSVs ──────────────────────────────────────
hima_files: dict[str, list[Path]] = {}
for f in HIMA_DIR.glob('*.csv'):
    hima_files.setdefault(f.stem, []).append(f)

# Find stations present in all three sources
master_stations = set(_meta[name_col].tolist())
mapped_sids     = set(station_id_map.values())
common_ids      = sorted(mapped_sids & set(hima_files) & master_stations)

print(f'Himawari station files  : {len(hima_files)}')
print(f'Masterdata stations     : {len(master_stations)}')
print(f'VIIRS→masterdata mapped : {len(mapped_sids)}')
print(f'Common (all three)      : {len(common_ids)}')
print('\nFirst 10 common IDs:')
for sid in common_ids[:10]:
    print(' ', sid)

## 2 · Loader Functions

In [ ]:
def load_viirs_station(station_label: str) -> pd.DataFrame:
    """
    Extract VIIRS overpasses for a single station from the pre-loaded flat CSV.
    Returns a DataFrame indexed by datetime with column AOD_VIIRS_055.
    """
    df = viirs_raw[viirs_raw['station'] == station_label].copy()
    df = df.sort_values('datetime').set_index('datetime')

    out = pd.DataFrame(index=df.index)
    out[VIIRS_COL]      = df['aod'].astype(float)
    out['QA_VIIRS']     = df['qa']
    out['source_sds']   = df['source_sds']
    out['station_lat']  = df['station_lat']
    out['station_lon']  = df['station_lon']
    return out


def load_himawari(paths: list[Path]) -> pd.DataFrame:
    """
    Load and concatenate Himawari L3 station CSVs for one station.
    Returns:
      - AOD_Himawari  : AOT_Merged_center at 0.50 µm
      - AE_Himawari   : Ångström exponent from AE_Merged (if present)
      - QA_Himawari   : QA flag (if present)
    """
    frames = [pd.read_csv(p, parse_dates=['timestamp'], low_memory=False) for p in paths]
    df = pd.concat(frames, ignore_index=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.dropna(subset=['timestamp'])
    df = df.sort_values('timestamp').set_index('timestamp')

    out = pd.DataFrame({HIMA_050_COL: df['AOT_Merged_center'].astype(float)})

    if 'AE_Merged' in df.columns:
        out['AE_Himawari'] = df['AE_Merged'].astype(float)
    if 'QA_flag_Merged' in df.columns:
        out['QA_Himawari'] = df['QA_flag_Merged']

    return out


print('Loader functions defined ✓')

## 3 · Build Merged Dataset (all stations)

For each VIIRS overpass we average all Himawari observations within a **±1 h window**.  
We also average `AE_Himawari` over that window and use it to compute `AOD_Himawari_055`:

- **AE available** → `AOD_Himawari_055 = AOD_Himawari × (0.55 / 0.50)^(−AE)`  
- **AE = NaN** → `AOD_Himawari_055 = AOD_Himawari` (keep original 0.50 µm value)

In [ ]:
records = []
station_dfs = {}   # per-station merged DF for later cells

# Reverse map: masterdata stationName → VIIRS station label
rev_map = {v: k for k, v in station_id_map.items()}

_radius = pd.Timedelta('1h')

for sid in common_ids:
    viirs_label = rev_map.get(sid)
    if viirs_label is None:
        continue
    try:
        viirs = load_viirs_station(viirs_label)
        hima  = load_himawari(hima_files[sid])

        if len(viirs) == 0:
            continue

        hima_aod = hima[HIMA_050_COL]
        has_ae   = 'AE_Himawari' in hima.columns
        has_qa   = 'QA_Himawari' in hima.columns

        # ── Make Himawari index timezone-naive for safe slicing ─────────────
        if hima_aod.index.tz is not None:
            hima_aod.index = hima_aod.index.tz_localize(None)
            hima.index = hima.index.tz_localize(None)

        # ── Make VIIRS index timezone-naive ─────────────────────────────────
        if viirs.index.tz is not None:
            viirs.index = viirs.index.tz_localize(None)

        aod_means, ae_means, qa_modes = [], [], []

        for ts in viirs.index:
            t0, t1 = ts - _radius, ts + _radius
            window_aod = hima_aod.loc[t0:t1]
            aod_means.append(window_aod.mean() if len(window_aod) > 0 else float('nan'))

            if has_ae:
                window_ae = hima['AE_Himawari'].loc[t0:t1]
                ae_means.append(window_ae.mean() if len(window_ae) > 0 else float('nan'))

            if has_qa:
                window_qa = hima['QA_Himawari'].loc[t0:t1]
                mode_val  = window_qa.mode()
                qa_modes.append(mode_val.iloc[0] if len(mode_val) > 0 else float('nan'))

        merged = viirs.copy()
        merged[HIMA_050_COL] = aod_means

        if has_ae:
            merged['AE_Himawari'] = ae_means
        else:
            merged['AE_Himawari'] = float('nan')

        if has_qa:
            merged['QA_Himawari'] = qa_modes

        # ── Wavelength correction ────────────────────────────────────────────
        ae_col = merged['AE_Himawari']
        merged[HIMA_055_COL] = np.where(
            ae_col.notna(),
            merged[HIMA_050_COL] * WL_RATIO ** (-ae_col),
            merged[HIMA_050_COL]
        )

        # Drop rows where no Himawari data fell in the ±1 h window
        merged = merged.dropna(subset=[HIMA_050_COL])
        if len(merged) == 0:
            continue

        merged['station_id']    = sid
        merged['station_label'] = viirs_label

        station_dfs[sid] = merged
        records.append(merged.reset_index().rename(columns={'index': 'timestamp', 'datetime': 'timestamp'}))

    except Exception as e:
        print(f'[WARN] {sid}: {e}')

# ── Consolidate ──────────────────────────────────────────────────────────────
all_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()

# Normalise the timestamp column name across concat frames
if 'datetime' in all_df.columns and 'timestamp' not in all_df.columns:
    all_df = all_df.rename(columns={'datetime': 'timestamp'})
all_df['timestamp'] = pd.to_datetime(all_df['timestamp'])

n_ae_converted = all_df['AE_Himawari'].notna().sum()
n_fallback      = all_df['AE_Himawari'].isna().sum()

print(f'Total matched records       : {len(all_df):,}')
print(f'Stations with data          : {all_df["station_id"].nunique()}')
print(f'Date range                  : {all_df["timestamp"].min().date()} → {all_df["timestamp"].max().date()}')
print(f'AOD_Himawari_055 (AE-conv.) : {n_ae_converted:,} rows  '
      f'({100*n_ae_converted/max(len(all_df),1):.1f}%)')
print(f'AOD_Himawari_055 (fallback) : {n_fallback:,} rows  '
      f'({100*n_fallback/max(len(all_df),1):.1f}%)')
all_df[[VIIRS_COL, HIMA_050_COL, HIMA_055_COL, 'AE_Himawari']].head()

## 4 · Data Overview & Missing-Value Summary

In [ ]:
print('=== Basic statistics ===')
display(
    all_df[[VIIRS_COL, HIMA_050_COL, HIMA_055_COL, 'AE_Himawari']]
    .describe().T.round(4)
)

pair_counts = all_df.groupby('station_id').size().rename('n_pairs').sort_values(ascending=False)
print(f'\nMedian pairs per station : {pair_counts.median():.0f}')
print(f'Stations with ≥30 pairs  : {(pair_counts >= 30).sum()}')

fig, ax = plt.subplots(figsize=(10, 3))
pair_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Number of co-located overpass pairs per station')
ax.set_xlabel('Station ID'); ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6)
plt.tight_layout(); plt.show()

## 5 · AOD Value Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

series_cfg = [
    (VIIRS_COL,    '#C4782A', 'VIIRS 0.55 µm (NOAA-20)'),
    (HIMA_050_COL, '#2D7DBB', 'Himawari 0.50 µm (original)'),
    (HIMA_055_COL, '#1B4F8A', 'Himawari 0.55 µm (AE-corrected)'),
]

# Histogram
for col, color, label in series_cfg:
    all_df[col].plot.hist(bins=60, alpha=0.45, color=color, label=label, ax=axes[0])
axes[0].set_xlabel('AOD'); axes[0].set_title('Distribution of AOD values')
axes[0].legend(fontsize=8)

# KDE
for col, color, label in series_cfg:
    all_df[col].dropna().plot.kde(ax=axes[1], color=color, lw=2, label=label)
axes[1].set_xlabel('AOD'); axes[1].set_title('Kernel Density Estimate')
axes[1].legend(fontsize=8)

# Box plots
all_df[[c for c, _, _ in series_cfg]].rename(columns={
    VIIRS_COL:    'VIIRS 0.55',
    HIMA_050_COL: 'Hima 0.50',
    HIMA_055_COL: 'Hima 0.55'
}).plot.box(
    ax=axes[2],
    color={'boxes': 'steelblue', 'medians': 'red', 'whiskers': 'gray', 'caps': 'gray'},
    patch_artist=True
)
axes[2].set_title('Box plots'); axes[2].set_ylabel('AOD')

plt.suptitle('AOD Value Distributions — All Stations', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

# AE distribution
fig, ax = plt.subplots(figsize=(6, 3))
all_df['AE_Himawari'].dropna().plot.hist(bins=50, color='#9B59B6', edgecolor='white', ax=ax)
ax.set_xlabel('Ångström Exponent (AE)')
ax.set_title('Distribution of AE_Merged (used for Himawari wavelength correction)')
plt.tight_layout(); plt.show()

## 6 · Global Correlation Analysis

Side-by-side: **VIIRS 0.55 µm vs Himawari 0.50 µm (original)** and **vs Himawari 0.55 µm (AE-corrected)**.

In [ ]:
def regression_stats(x: pd.Series, y: pd.Series) -> dict:
    """Compute key regression / agreement statistics (x = reference, y = estimate)."""
    mask = x.notna() & y.notna()
    x, y = x[mask].values, y[mask].values
    if len(x) < 3:
        return dict(n=len(x), r=np.nan, R2=np.nan, slope=np.nan,
                    intercept=np.nan, RMSE=np.nan, MAE=np.nan,
                    Bias=np.nan, RelBias_pct=np.nan)
    slope, intercept, r, p, _ = stats.linregress(x, y)
    rmse    = np.sqrt(mean_squared_error(x, y))
    mae     = mean_absolute_error(x, y)
    bias    = np.mean(y - x)          # Himawari − VIIRS
    relbias = bias / np.mean(x) * 100
    return dict(n=len(x), r=r, R2=r**2, slope=slope, intercept=intercept,
                RMSE=rmse, MAE=mae, Bias=bias, RelBias_pct=relbias)


# ── 2 × 2 grid: [scatter | residuals] for each Himawari variant ─────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
gs_dict = {}

for i, (hima_col, hima_label, color) in enumerate(HIMA_VARIANTS):
    gs = regression_stats(all_df[VIIRS_COL], all_df[hima_col])
    gs_dict[hima_col] = gs

    # ── Scatter ───────────────────────────────────────────────────────────────
    ax = axes[i, 0]
    valid = all_df[[VIIRS_COL, hima_col]].dropna()
    ax.scatter(valid[VIIRS_COL], valid[hima_col],
               c=color, edgecolors='white', linewidths=0.4, s=35, alpha=0.75)

    q99  = max(valid[VIIRS_COL].quantile(0.99), valid[hima_col].quantile(0.99))
    lims = [0, q99 * 1.05]
    xfit = np.linspace(*lims, 200)
    ax.plot(lims, lims, 'k--', lw=1.2, label='1:1 line')
    ax.plot(xfit, gs['slope']*xfit + gs['intercept'], 'b-', lw=2,
            label=f"y = {gs['slope']:.3f}x {gs['intercept']:+.3f}")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('VIIRS 0.55 µm AOD (NOAA-20)', fontsize=10)
    ax.set_ylabel(hima_label, fontsize=10)
    ax.set_title(f'VIIRS 0.55 vs {hima_label}\n(n={gs["n"]:,})', fontweight='bold')
    stats_text = (f"R² = {gs['R2']:.3f}   r = {gs['r']:.3f}\n"
                  f"RMSE = {gs['RMSE']:.3f}   MAE = {gs['MAE']:.3f}\n"
                  f"Bias = {gs['Bias']:+.3f}   RelBias = {gs['RelBias_pct']:+.1f}%")
    ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
            va='top', ha='left', fontsize=9, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.9))
    ax.legend(fontsize=8, loc='lower right')

    # ── Residuals ──────────────────────────────────────────────────────────────
    ax2 = axes[i, 1]
    resid = all_df[hima_col] - all_df[VIIRS_COL]
    valid_resid = all_df[[VIIRS_COL]].dropna().join(resid.rename('resid')).dropna()

    pos = valid_resid['resid'] >= 0
    ax2.scatter(valid_resid.loc[pos, VIIRS_COL], valid_resid.loc[pos, 'resid'],
                c='#2D7DBB', s=30, alpha=0.7, edgecolors='white', lw=0.3,
                label='Himawari > VIIRS')
    ax2.scatter(valid_resid.loc[~pos, VIIRS_COL], valid_resid.loc[~pos, 'resid'],
                c='#E07B39', s=30, alpha=0.7, edgecolors='white', lw=0.3,
                label='Himawari < VIIRS')
    ax2.axhline(0,          color='k',   lw=1.2, linestyle='--')
    ax2.axhline(gs['Bias'], color='red', lw=1.5, linestyle='-',
                label=f"Mean bias = {gs['Bias']:+.3f}")
    ax2.axhspan(-gs['RMSE'], gs['RMSE'], color='gray', alpha=0.08, label='±RMSE band')
    ax2.set_xlabel('VIIRS 0.55 µm AOD', fontsize=10)
    ax2.set_ylabel('Himawari − VIIRS  (residual)', fontsize=10)
    ax2.set_title(f'Residuals — {hima_label}', fontweight='bold')
    ax2.legend(fontsize=8)

plt.suptitle(
    'Global Correlation — All Stations Combined\n'
    'Upper row: vs Himawari 0.50 µm (original)  |  '
    'Lower row: vs Himawari 0.55 µm (AE-corrected)',
    fontweight='bold', fontsize=12
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

pd.DataFrame(gs_dict).T.round(4)

## 7 · Per-Station Correlation Statistics

In [ ]:
station_stats = []
for sid, df in station_dfs.items():
    if len(df) < 5:
        continue
    for hima_col, hima_label, _ in HIMA_VARIANTS:
        s = regression_stats(df[VIIRS_COL], df[hima_col])
        s['station_id']   = sid
        s['hima_variant'] = hima_label
        s['station_label'] = df['station_label'].iloc[0] if 'station_label' in df.columns else sid
        station_stats.append(s)

stats_df = (
    pd.DataFrame(station_stats)
    .set_index(['station_id', 'hima_variant'])
    .round(4)
)

print(f'Stations with ≥5 pairs: {stats_df.index.get_level_values(0).nunique()}')
display(
    stats_df
    .sort_values(['hima_variant', 'R2'], ascending=[True, False])
    .head(40)
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
subplot_labels = string.ascii_lowercase
plot_idx = 0

hima_row_cfg = [
    (HIMA_050_COL, '0.50 µm (Orig.)'),
    (HIMA_055_COL, '0.55 µm (AE-corr.)'),
]

for row, (hima_col, hima_label) in enumerate(hima_row_cfg):
    variant_label = next(lbl for col, lbl, _ in HIMA_VARIANTS if col == hima_col)
    band_stats = stats_df.xs(variant_label, level='hima_variant')

    colors = ['#4C72B0', '#DD8452', '#55A868']
    for col_idx, (metric, color, xlabel) in enumerate([
        ('R2',   colors[0], 'R²'),
        ('RMSE', colors[1], 'RMSE'),
        ('Bias', colors[2], 'Bias'),
    ]):
        ax = axes[row, col_idx]
        data = band_stats[metric].dropna()

        ax.hist(data, bins=min(15, len(data)), color=color, edgecolor='black', linewidth=0.7, alpha=0.85)

        median_val = data.median()
        ax.axvline(median_val, color='#cc0000', lw=1.5, linestyle='--')

        if metric == 'Bias':
            ax.axvline(0, color='black', lw=1.5, linestyle=':')

        ax.text(0.95, 0.90, f'Median: {median_val:.3f}',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=11, fontweight='bold', color='#cc0000',
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1))

        if row == 1:
            ax.set_xlabel(xlabel, weight='bold')

        if col_idx == 0:
            ax.set_ylabel('Number of stations', weight='bold')

        ax.set_title(f'({subplot_labels[plot_idx]}) {metric} — {hima_label}',
                     loc='left', fontweight='bold', pad=8)
        sns.despine(ax=ax)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4, color='gray')
        plot_idx += 1

plt.tight_layout()
plt.show()

## 8 · Seasonal / Monthly Correlation

In [ ]:
all_df['month'] = all_df['timestamp'].dt.month
all_df['year']  = all_df['timestamp'].dt.year

monthly_dfs = {}
for hima_col, hima_label, color in HIMA_VARIANTS:
    monthly_stats = []
    for m, grp in all_df.groupby('month'):
        if len(grp) < 10:
            continue
        s = regression_stats(grp[VIIRS_COL], grp[hima_col])
        s['month'] = m
        monthly_stats.append(s)
    monthly_dfs[hima_col] = pd.DataFrame(monthly_stats).set_index('month')

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, col, ylabel, title in [
    (axes[0,0], 'R2',          'R²',                'Monthly R²'),
    (axes[0,1], 'RMSE',        'RMSE',              'Monthly RMSE'),
    (axes[1,0], 'Bias',        'Bias (Hima−VIIRS)', 'Monthly Bias'),
    (axes[1,1], 'RelBias_pct', 'Relative Bias (%)', 'Monthly Relative Bias'),
]:
    for hima_col, hima_label, color in HIMA_VARIANTS:
        df_m = monthly_dfs[hima_col]
        ax.plot(df_m.index, df_m[col], marker='o', color=color, label=hima_label)
    ref_months = monthly_dfs[HIMA_050_COL].index
    ax.set_xticks(ref_months)
    ax.set_xticklabels([month_names[m-1] for m in ref_months])
    ax.set_title(title); ax.set_ylabel(ylabel)
    if col == 'Bias':
        ax.axhline(0, color='black', lw=1, linestyle='--')
    ax.legend(fontsize=8)

plt.suptitle('Seasonal Variation — VIIRS 0.55 µm (NOAA-20) vs Himawari variants', fontweight='bold')
plt.tight_layout(); plt.show()

## 9 · Time-Series Comparison — Top-N Stations

In [ ]:
TOP_N = 13

n_by_station = all_df.groupby('station_id').size().sort_values(ascending=False)
top_stations = n_by_station.head(TOP_N).index.tolist()

fig, axes = plt.subplots(TOP_N, 1, figsize=(16, 3.8 * TOP_N), sharex=False)
if TOP_N == 1:
    axes = [axes]

for ax, sid in zip(axes, top_stations):
    df   = station_dfs[sid]
    name = df['station_label'].iloc[0] if 'station_label' in df.columns else sid

    ax.plot(df.index, df[VIIRS_COL],    's-',  color='#C4782A', ms=4, lw=1.3,
            label='VIIRS 0.55 µm (NOAA-20)', alpha=0.9)
    ax.plot(df.index, df[HIMA_050_COL], 'o-',  color='#2D7DBB', ms=4, lw=1.5,
            label='Himawari 0.50 µm (original)', alpha=0.9)
    ax.plot(df.index, df[HIMA_055_COL], '^--', color='#1B4F8A', ms=4, lw=1.5,
            label='Himawari 0.55 µm (AE-corrected)', alpha=0.9)

    ax.fill_between(df.index, df[VIIRS_COL], df[HIMA_050_COL],
                    alpha=0.07, color='gray')

    def _r2(sid, hima_label):
        key = (sid, hima_label)
        return f"{stats_df.loc[key, 'R2']:.3f}" if key in stats_df.index else 'N/A'

    lbl_050 = next(lbl for col, lbl, _ in HIMA_VARIANTS if col == HIMA_050_COL)
    lbl_055 = next(lbl for col, lbl, _ in HIMA_VARIANTS if col == HIMA_055_COL)
    n = df[[VIIRS_COL, HIMA_050_COL]].dropna().shape[0]

    ax.set_title(
        f"{str(name)[:65]}\n"
        f"R²(vs 0.50)={_r2(sid, lbl_050)}  |"
        f"  R²(vs 0.55 AE)={_r2(sid, lbl_055)}  |  n={n} co-located overpasses",
        fontsize=9
    )
    ax.set_ylabel('AOD')
    ax.legend(fontsize=8, loc='upper right', ncol=3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.set_ylim(bottom=0)

plt.suptitle(
    f'AOD Time-Series — Top {TOP_N} Stations by Data Count\n'
    'Brown = VIIRS 0.55 µm (NOAA-20)  |  Blue = Himawari 0.50 µm  |  Dark Blue = Himawari 0.55 µm (AE-corrected)',
    fontweight='bold', fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

## 10 · Interactive Station Selector

In [ ]:
STATION_ID = top_stations[0]   # ← replace with any station_id string from common_ids

df   = station_dfs[STATION_ID]
name = df['station_label'].iloc[0] if 'station_label' in df.columns else STATION_ID

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for row, (hima_col, hima_label, color) in enumerate(HIMA_VARIANTS):
    s = regression_stats(df[VIIRS_COL], df[hima_col])

    # ── Time-series ──────────────────────────────────────────────────────────
    ax = axes[row, 0]
    ax.plot(df.index, df[VIIRS_COL], 's-', color='#C4782A', ms=5, lw=1.5,
            label='VIIRS 0.55 µm (NOAA-20)')
    ax.plot(df.index, df[hima_col],  'o-', color=color,     ms=5, lw=1.5,
            label=hima_label)
    ax.fill_between(df.index, df[VIIRS_COL], df[hima_col], alpha=0.12, color='gray')
    ax.set_title(f'Time-Series — VIIRS 0.55 vs {hima_label}', fontweight='bold')
    ax.set_ylabel('AOD')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.set_ylim(bottom=0)

    # ── Scatter ──────────────────────────────────────────────────────────────
    ax2 = axes[row, 1]
    valid = df[[VIIRS_COL, hima_col]].dropna()
    ax2.scatter(valid[VIIRS_COL], valid[hima_col],
                c=color, edgecolors='white', linewidths=0.5, s=60, alpha=0.85, zorder=3)
    lim_max = max(valid[VIIRS_COL].max(), valid[hima_col].max()) * 1.08
    lims    = [0, lim_max]
    xfit    = np.linspace(*lims, 200)
    ax2.plot(lims, lims, 'k--', lw=1.2, label='1:1 line')
    ax2.plot(xfit, s['slope']*xfit + s['intercept'], 'r-', lw=2,
             label=f"y={s['slope']:.3f}x {s['intercept']:+.3f}")
    ax2.set_xlim(lims); ax2.set_ylim(lims)
    ax2.set_aspect('equal', adjustable='box')
    ax2.set_xlabel('VIIRS 0.55 µm AOD (NOAA-20)', fontsize=10)
    ax2.set_ylabel(hima_label, fontsize=10)
    ax2.set_title(f'Scatter — VIIRS 0.55 vs {hima_label}', fontweight='bold')
    stats_text = (f"n = {s['n']}   R² = {s['R2']:.3f}\n"
                  f"RMSE = {s['RMSE']:.3f}   Bias = {s['Bias']:+.3f}")
    ax2.text(0.03, 0.97, stats_text, transform=ax2.transAxes,
             va='top', fontsize=10, fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.9))
    ax2.legend(fontsize=9, loc='lower right')

plt.suptitle(
    f'Detailed View — {str(name)[:70]}\n'
    'Upper row: vs Himawari 0.50 µm (original)  |  '
    'Lower row: vs Himawari 0.55 µm (AE-corrected)',
    fontweight='bold', fontsize=11
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 11 · Global Time-Series (Monthly Mean, All Stations)

In [ ]:
cols_to_plot = [VIIRS_COL, HIMA_050_COL, HIMA_055_COL]

monthly_mean = (
    all_df.set_index('timestamp')[cols_to_plot]
    .resample('1ME')
    .agg(['mean', 'std'])
)
monthly_mean.columns = ['_'.join(c) for c in monthly_mean.columns]

fig, ax = plt.subplots(figsize=(16, 5))

plot_cfg = [
    (VIIRS_COL,    '#C4782A', 'VIIRS 0.55 µm (NOAA-20)',          '-'),
    (HIMA_050_COL, '#2D7DBB', 'Himawari 0.50 µm (original)',       '-'),
    (HIMA_055_COL, '#1B4F8A', 'Himawari 0.55 µm (AE-corrected)',  '--'),
]

for col, color, label, ls in plot_cfg:
    mu  = monthly_mean[f'{col}_mean']
    sig = monthly_mean[f'{col}_std']
    ax.plot(mu.index, mu, color=color, lw=2.2, linestyle=ls, label=label, zorder=3)
    ax.fill_between(mu.index, (mu - sig).clip(lower=0), mu + sig,
                    color=color, alpha=0.10)

ax.set_title(
    'Monthly Mean AOD ± 1σ — All Stations\n'
    'Shaded band = ±1 standard deviation across stations',
    fontweight='bold'
)
ax.set_ylabel('AOD')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 12 · Heatmap — R² by Station × Month

In [ ]:
MIN_MONTHLY_N = 3

station_n       = stats_df.groupby(level='station_id')['n'].max()
HEATMAP_STATIONS = station_n[station_n >= 10].sort_values(ascending=False).head(15).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(22, max(4, len(HEATMAP_STATIONS) * 0.6)))

for ax, (hima_col, hima_label, _) in zip(axes, HIMA_VARIANTS):
    pivot_r2 = pd.DataFrame(index=HEATMAP_STATIONS, columns=range(1, 13), dtype=float)

    for sid in HEATMAP_STATIONS:
        df_s = station_dfs[sid].copy()
        df_s['month'] = df_s.index.month
        for m, grp in df_s.groupby('month'):
            valid = grp[[VIIRS_COL, hima_col]].dropna()
            if len(valid) >= MIN_MONTHLY_N:
                r, _ = stats.pearsonr(valid[VIIRS_COL], valid[hima_col])
                pivot_r2.loc[sid, m] = round(r**2, 3)

    pivot_r2.columns = [month_names[m-1] for m in pivot_r2.columns]
    pivot_r2.index   = [str(s)[-8:] for s in pivot_r2.index]

    sns.heatmap(
        pivot_r2.astype(float), annot=True, fmt='.2f',
        cmap='RdYlGn', vmin=0, vmax=1,
        linewidths=0.5, ax=ax, cbar_kws={'label': 'R²'},
        annot_kws={'size': 8}
    )
    ax.set_title(f'R² — VIIRS 0.55 vs {hima_label}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Month')
    ax.set_ylabel('Station (last 8 chars of ID)')

plt.suptitle(
    'Monthly R² by Station\n'
    'Left: vs Himawari 0.50 µm (original)  |  '
    'Right: vs Himawari 0.55 µm (AE-corrected)  '
    '(green = strong agreement, red = poor)',
    fontweight='bold', fontsize=12
)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 13 · Summary Table — Export

In [ ]:
export_path = 'AOD_VIIRS_Himawari_station_correlation_summary.csv'
stats_df.sort_values(['hima_variant', 'R2'], ascending=[True, False]).to_csv(export_path)
print(f'Saved → {export_path}')

# Per-variant global stats
gs_050 = regression_stats(all_df[VIIRS_COL], all_df[HIMA_050_COL])
gs_055 = regression_stats(all_df[VIIRS_COL], all_df[HIMA_055_COL])

lbl_050 = next(lbl for col, lbl, _ in HIMA_VARIANTS if col == HIMA_050_COL)
lbl_055 = next(lbl for col, lbl, _ in HIMA_VARIANTS if col == HIMA_055_COL)
s050 = stats_df.xs(lbl_050, level='hima_variant')
s055 = stats_df.xs(lbl_055, level='hima_variant')

summary = {
    'Metric': [
        'N co-located records',
        'N stations',
        'VIIRS QA threshold applied',
        'Records with Himawari AE correction',
        'Records using Himawari 0.50 fallback (AE = NaN)',
        'Median R²  — vs Himawari 0.50 µm',
        'Median R²  — vs Himawari 0.55 µm (AE)',
        'Median RMSE — vs Himawari 0.50 µm',
        'Median RMSE — vs Himawari 0.55 µm (AE)',
        'Global Bias — vs Himawari 0.50 µm  (Hima−VIIRS)',
        'Global Bias — vs Himawari 0.55 µm  (Hima−VIIRS)',
        'Stations R²≥0.7 — vs Himawari 0.50 µm',
        'Stations R²≥0.7 — vs Himawari 0.55 µm (AE)',
    ],
    'Value': [
        f"{len(all_df):,}",
        f"{all_df['station_id'].nunique()}",
        f"qa >= {VIIRS_QA_MIN}",
        f"{all_df['AE_Himawari'].notna().sum():,}  "
        f"({100*all_df['AE_Himawari'].notna().mean():.1f}%)",
        f"{all_df['AE_Himawari'].isna().sum():,}  "
        f"({100*all_df['AE_Himawari'].isna().mean():.1f}%)",
        f"{s050['R2'].median():.3f}",
        f"{s055['R2'].median():.3f}",
        f"{s050['RMSE'].median():.3f}",
        f"{s055['RMSE'].median():.3f}",
        f"{gs_050['Bias']:+.3f}  ({gs_050['RelBias_pct']:+.1f}%)",
        f"{gs_055['Bias']:+.3f}  ({gs_055['RelBias_pct']:+.1f}%)",
        f"{(s050['R2'] >= 0.7).sum()} / {len(s050)}",
        f"{(s055['R2'] >= 0.7).sum()} / {len(s055)}",
    ]
}
display(pd.DataFrame(summary).set_index('Metric'))